#### GPU Scaling 

After the successful implementation of Yann's ideas in 1998 

In [6]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP
import os
import torch.nn as nn
import torch.nn.functional as F

# set the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# set the seed
torch.manual_seed(42) # for reproducibility

In [7]:
class LocalResponseNorm(nn.Module):
    """
    local response normalization was used before batch normalization, it normalizes over local neighborhoods in feature maps, creating completion 
    betweeen neurons outputs computed using different kernels.
    """
    def __init__(self, size: int=5, alpha: float = 0.0001, beta: float = 0.75, k: float = 2.0):
        super(LocalResponseNorm, self).__init__()
        self.size = size
        self.alpha = alpha
        self.beta = beta
        self.k = k

    def forward(self, input: torch.Tensor) -> torch.Tensor:
        return F.local_response_norm(input, self.size, self.alpha, self.beta, self.k)

In [8]:
class AlexNetRepresentationLearner(nn.Module):
    """
    convolutional layers of the alexnet
    the original paper used two gpus and split the network across them
    certain layers had cross-gpu connections while others did not
    here we implement the full network into a single gpu but note where the split occurs
    """
    def __init__(self):
        super(AlexNetRepresentationLearner, self).__init__()
        # layer 1: 96 kernels of size 11*11*3 witih stride 4
        # original paper split into 48 kernels per gpu
        # input is 224*224*3 -> 55*55*96
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=96, kernel_size=11, stride=4, padding=5)
        self.lrn1 = LocalResponseNorm()
        # max pooling: 3*3 with stride 2
        # input is 55*55*96 -> 27*27*96
        self.maxpool1 = nn.MaxPool2d(kernel_size=3, stride=2)

        # layer 2: 256 kernels of size 5*5*48 (with cross-gpu connections)
        # original paper split into 128 kernels per gpu taking input from both gpus
        # input is 27*27*96 -> 27*27*256
        self.conv2 = nn.Conv2d(in_channels=96, out_channels=256, kernel_size=5, padding=2)
        self.lrn2 = LocalResponseNorm()
        # max pooling: 3*3 with stride 2
        # input is 27*27*256 -> 13*13*256
        self.pool2 = nn.MaxPool2d(kernel_size=3, stride=2)

        # layer 3: 384 kernels of size 3*3*256 (with cross-gpu connections)
        # original paper split: 192 kernels per gpu, taking input from the same gpu only
        # input is 13*13*256 -> 13*13*384
        self.conv3 = nn.Conv2d(in_channels=256, out_channels=384, kernel_size=3, padding=1)
        # layer 4: 384 kernels of size 3*3*192 (with cross-gpu connections)
        # original paper split: 192 kernels per gpu, taking input from the same gpu only
        # input is 13*13*384 -> 13*13*384
        self.conv4 = nn.Conv2d(in_channels=384, out_channels=384, kernel_size=3, padding=1)
        # layer 5: 256 kernels of size 3*3*384 (with cross-gpu connections)
        # original paper split: 128 kernels per gpu, taking input from the same gpu only
        # input is 13*13*384 -> 6*6*256
        self.conv5 = nn.Conv2d(in_channels=384, out_channels=256, kernel_size=3, padding=1)
        # max pooling: 3*3 with stride 2
        # input is 6*6*256 -> 3*3*256
        self.pool5 = nn.MaxPool2d(kernel_size=3, stride=2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # conv1 + relu + lrn1 + maxpool1
        # conv1: 224*224*3 -> 55*55*96
        x = F.relu(self.conv1(x))
        # lrn1
        x = self.lrn1(x)
        # maxpool1: 55*55*96 -> 27*27*96
        x = self.maxpool1(x)

        # conv2 + relu + lrn2 + maxpool2
        # conv2: 27*27*96 -> 27*27*256
        x = F.relu(self.conv2(x))
        # lrn2
        x = self.lrn2(x)
        # maxpool2: 27*27*256 -> 13*13*256
        x = self.pool2(x)

        # conv3 + relu
        # conv3: 13*13*256 -> 13*13*384
        x = F.relu(self.conv3(x))
        # conv4 + relu
        # conv4: 13*13*384 -> 13*13*384
        x = F.relu(self.conv4(x))
        # conv5 + relu + maxpool5
        # conv5: 13*13*384 -> 6*6*256
        x = F.relu(self.conv5(x))
        # maxpool5: 6*6*256 -> 3*3*256
        x = self.pool5(x)
    
        return x

In [9]:
class AlexNetClassifier(nn.Module):
    """
    the fully connected classification layers of the alexnet
    the original paper uses dropwout with p=0.5 in the first two fc layers
    to prevent overfitting. this is one fo the first prominent use of dropout in deep learning.
    """

    def __init__(self, num_classes: int=1000, dropout:float=0.5):
        super(AlexNetClassifier, self).__init__()
        # fc6: 4096 neurons
        # input is 6*6*256 = 9216 output: 4096
        self.fc6 = nn.Linear(in_features=(6*6*256), out_features=4096)
        self.dropout6 = nn.Dropout(p=dropout)

        # fc7: 4096 neurons
        # input is 4096 output: 4096
        self.fc7 = nn.Linear(in_features=4096, out_features=4096)
        self.dropout7 = nn.Dropout(p=dropout)

        # fc8: num_classes neurons
        # input is 4096 output: num_classes
        self.fc8 = nn.Linear(in_features=4096, out_features=num_classes)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # flatten the output of the last convolutional layer
        x = torch.flatten(x, 1)
        # fc6
        x = F.relu(self.fc6(x))
        x = self.dropout6(x)
        # fc7
        x = F.relu(self.fc7(x))
        x = self.dropout7(x)

        # fc8
        x = self.fc8(x)
        return x

In [10]:
class AlexNet(nn.Module):
    """
    complete alexnet architecture
    this implementation is the exact architecture from the 2012 paper
    - 5 convulational layers witih relu activation
    - max pooling at end of convlutations 1, 2 and 5
    - 3 fully connected layers
    - dropout at first two fc layers to prevent overfitting
    """

    def __init__(self, num_classes: int=1000, dropout: float=0.5):
        super(AlexNet, self).__init__()

        self.features = AlexNetRepresentationLearner()
        self.classifier = AlexNetClassifier()

        self._initialize_weights()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.classifier(x)
        return x
    
    def _initialize_weights(self):
        """
        initialization of the weights of the model as described by the original paper
        - conv and fc weights: gaussian distributed with mean 0 and std 0.01
        - conv biases: 0 for layers 1, 3, 5 and 1 for layers 2, 4
        - fc biases: 1
        """
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.normal_(m.weight, 0, 0.01)
                # set biases based on layer number
                if m.out_channels == 96 and m.in_channels == 3: # i.e conv1
                    nn.init.constant_(m.bias, 0)
                elif m.out_channels ==  256 and m.in_channels == 96: # conv2
                    nn.init.constant_(m.bias, 1)
                elif m.out_channels == 384 and m.in_channels == 256: # conv3
                    nn.init.constant_(m.bias, 0)
                elif m.out_channels == 384 and m.in_channels == 384: # conv4
                    nn.init.constant_(m.bias, 1)
                elif m.out_channels == 256 and m.in_channels == 384: # conv5
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 1)

In [11]:
def get_imagenet_transforms(is_training: bool=True):
    """
    data augmentation and preprocessing as described in the alexnet paper
    training augmentations:
    - random crop of 224 from 256 images
    - random horizontal flip
    - pca-based color augmentation
    - normalization
    """
    if is_training:
        return transforms.Compose([
            transforms.Resize(256),
            transforms.RandomCrop(224),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
            transforms.RandomGrayscale(p=0.2),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    else:
        return transforms.Compose([
            transforms.Resize(224),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

In [12]:
class AlexNetTrainer:
    """
    training configuration and loop following the original paper methology

    key details of implementation:
    - SGD with momentum (0.9)
    - weight decay of 0.0005
    - learning rate of 0.01, divided by 10 when validation error plateaus
    - batch isze: 128
    - trained for ~90 epochs
    """

    def __init__(self,
                 model: nn.Module, # alexnet model
                 device: torch.device, # device to train on
                 train_loader: DataLoader, # training data loader
                 val_loader: DataLoader, # validation data loader
                 learning_rate: float=0.01, # initial learning rate
                 weight_decay: float=0.0005, # weight decay
                 momentum: float=0.9, # momentum
                 lr_patience: int=10, # number of epochs to wait before reducing learning rate
                 lr_factor: float=0.1, # factor by which to reduce learning rate
                 ):
        self.model = model
        self.device = device
        self.train_loader = train_loader
        self.val_loader = val_loader
        

        self.optimizer = torch.optim.SGD(
            model.parameters(),
            lr=learning_rate,
            momentum=momentum,
            weight_decay=weight_decay,
        )

        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer,
            mode='min',
            factor=lr_factor,
            patience=lr_patience,
            verbose=True,
        )

        self.criterion = nn.CrossEntropyLoss()

        self.train_losses = []
        self.val_losses = []
        self.val_accuracies = []

    def train_epoch(self):
        self.model.train()
        total_loss = 0.0
        num_batches = len(self.train_loader)

        for batch_idx, (images, labels) in enumerate(self.train_loader):
            data, target = data.to(self.device), target.to(self.device)
            # zero gradients
            self.optimizer.zero_grad()

            # forward pass
            output = self.model(data)
            loss = self.criterion(output, target)

            # backward pass
            loss.backward()
            self.optimizer.step()

            total_loss += loss.item()

            # print progress
            if batch_idx % 100 == 0:
                print(f"batch {batch_idx} of {num_batches} | loss: {total_loss / (batch_idx + 1):.4f}")
        avg_loss = total_loss / num_batches
        return avg_loss
    
    def validate(self):
        """
        validate the model
        """
        self.model.eval()
        total_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for data, target in self.val_loader:
                data, target = data.to(self.device), target.to(self.device)
                output = self.model(data)
                loss = self.criterion(output, target)
                total_loss += loss.item()
                _, predicted = torch.max(output.data, 1)
                total += target.size(0)
                correct += (predicted == target).sum().item()
        avg_loss = total_loss / len(self.val_loader)
        avg_accuracy = 100 * (correct / total)
        return avg_loss, avg_accuracy
    
    def save_checkpoint(self, epoch: int):
        """save the models checkpoints"""
        torch.save({
            'epoch': epoch,
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'scheduler_state_dict': self.scheduler.state_dict(),
            'train_losses': self.train_losses,
            'val_losses': self.val_losses,
            'val_accuracies': self.val_accuracies,
        }, f'checkpoint_epoch_{epoch}.pth')
    
    def train(self):
        """
        train the model
        """
        for epoch in range(self.num_epochs):
            train_loss = self.train_epoch()
            val_loss, val_accuracy = self.validate()

            # update learning rate
            self.scheduler.step(val_loss)

            self.train_losses.append(train_loss)
            self.val_losses.append(val_loss)
            self.val_accuracies.append(val_accuracy)
            
            print(f"Epoch {epoch+1}/{self.num_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Accuracy: {val_accuracy:.2f}%")

            if (epoch + 1) % 10 == 0:
                self.save_checkpoint(epoch + 1)
        


In [13]:
def setup_distributed(gpu_id: int, num_gpus: int):
    """ 
    setup distributed training
    """
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '12355'
    dist.init_process_group('nccl', 
                            rank=gpu_id, 
                            world_size=num_gpus)

def cleanup():
    """
    cleanup after training
    """
    if dist.is_initialized():   
        dist.destroy_process_group()


In [14]:
def train_distributed_worker(gpu_id:int, num_gpus:int):
    """
    worker function for distributed training on a single node
    the function runs on each gpu prcess. it is more efficient than
    data parallel because:
    - each process has its own python interpreter
    - better gpu memory utilization
    - more efficient gradient synchronization
    """
    # setup distributed training
    setup_distributed(gpu_id, num_gpus)

    torch.cuda.set_device(gpu_id)
    device = torch.device(f'cuda:{gpu_id}')
    if gpu_id == 0:
        print(f"training with distributed data parallel on {num_gpus} gpus")

    # get the model
    model = AlexNet(num_classes=1000, dropout=0.5)
    model = DDP(model, device_ids=[gpu_id])

    # data loading
    train_transforms = get_imagenet_transforms(is_training=True)
    val_transforms = get_imagenet_transforms(is_training=False)

    train_dataset = datasets.ImageNet(root='/mnt/data/datasets/imagenet/imagenet2012', 
                                      split='train', transform=train_transforms)
    val_dataset = datasets.ImageNet(root='/mnt/data/datasets/imagenet/imagenet2012', 
                                    split='val', transform=val_transforms)

    # setup distributed samplers
    train_sampler = torch.utils.data.distributed.DistributedSampler(train_dataset,
                                                                    num_replicas=num_gpus,
                                                                    rank=gpu_id,
                                                                    shuffle=True)
    val_sampler = torch.utils.data.distributed.DistributedSampler(val_dataset,
                                                                    num_replicas=num_gpus,
                                                                    rank=gpu_id)

    per_gpu_batch_size = 128 // num_gpus
    train_loader = DataLoader(dataset=train_dataset,
                              sampler=train_sampler,
                              batch_size=per_gpu_batch_size,
                              num_workers=4,
                              pin_memory=True)
    
    val_loader = DataLoader(dataset=val_dataset,
                            sampler=val_sampler,
                            batch_size=per_gpu_batch_size,
                            num_workers=4,
                            pin_memory=True)
    
    # create trainer
    trainer = AlexNetTrainer(model=model,
                             device=device,
                             train_loader=train_loader,
                             val_loader=val_loader,
                             num_epochs=90,
                             learning_rate=0.01,
                             weight_decay=0.0005,
                             momentum=0.9,
                             lr_patience=10,
                             lr_factor=0.1)
    trainer.train()

    # cleanup
    cleanup()

In [15]:

num_gpus = torch.cuda.device_count()
mp.spawn(
    train_distributed_worker,
    nprocs=num_gpus,
    args=(num_gpus,),
    join=True
)

Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/home/kaarl/miniconda3/envs/lgm-py310/lib/python3.10/multiprocessing/spawn.py", line 116, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/home/kaarl/miniconda3/envs/lgm-py310/lib/python3.10/multiprocessing/spawn.py", line 126, in _main
    self = reduction.pickle.load(from_parent)
AttributeError: Can't get attribute 'train_distributed_worker' on <module '__main__' (built-in)>
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/home/kaarl/miniconda3/envs/lgm-py310/lib/python3.10/multiprocessing/spawn.py", line 116, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/home/kaarl/miniconda3/envs/lgm-py310/lib/python3.10/multiprocessing/spawn.py", line 126, in _main
    self = reduction.pickle.load(from_parent)
AttributeError: Can't get attribute 'train_distributed_worker' on <module '__main__' (built-in)>


ProcessExitedException: process 1 terminated with exit code 1